# Micro-Hay event-aware training 02

Generates and caches a larger physiological four-compartment dataset, catalogues the teacher response, then compares MSE-GRU, event-aware GRU, Branch ELM, causal ConvGRU and causal ConvLSTM. Inputs are only binary presynaptic spikes; all 61 states are supervised; no teacher state is fed back.

In [ ]:
from pathlib import Path
import os, subprocess, sys
ROOT = Path('/kaggle/working/LearningSingleCompartiment')
if not (ROOT / 'pyproject.toml').exists():
    if ROOT.exists() and any(ROOT.iterdir()):
        raise RuntimeError(f'{ROOT} exists but is not the project; remove or rename it first')
    subprocess.check_call(['git', 'clone', 'https://github.com/Zagred47/LearningSingleCompartiment.git', str(ROOT)])
else:
    subprocess.check_call(['git', '-C', str(ROOT), 'pull', '--ff-only', 'origin', 'main'])
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '--no-build-isolation', '-e', str(ROOT)])
print('Project root:', ROOT)

## Configuration
Dataset generation and every epoch have progress bars. The HDF5 cache is reused on reruns. For a first smoke run set 8/2/2 trajectories, 2 epochs and select only `gru_mse,gru_event`.

In [ ]:
os.environ['HAY_EVENT_DATASET'] = '/kaggle/working/hay_micro_4c_event_enriched_v2.h5'
os.environ['HAY_EVENT_OUTPUT'] = '/kaggle/working/hay_micro_event_aware_02'
os.environ['HAY_EVENT_TRAIN_TRAJECTORIES'] = '48'
os.environ['HAY_EVENT_VALIDATION_TRAJECTORIES'] = '8'
os.environ['HAY_EVENT_TEST_TRAJECTORIES'] = '12'
os.environ['HAY_EVENT_DATA_WORKERS'] = '4'
os.environ['HAY_EVENT_EPOCHS'] = '30'
os.environ['HAY_EVENT_REPLAYS'] = '0'  # avoids double-counting; stratified windows are used below
os.environ['HAY_EVENT_WINDOWS_PER_EPOCH'] = '48'
os.environ['HAY_EVENT_REUSE_MODELS'] = '1'  # skip checkpoints already completed with the same dataset/config
os.environ['HAY_EVENT_MODELS'] = 'gru_mse,gru_event,branch_elm,conv_gru,conv_lstm'
%run /kaggle/working/LearningSingleCompartiment/notebooks/micro_event_aware_training_02.py

In [ ]:
from IPython.display import display
display(pd.read_csv('/kaggle/working/hay_micro_event_aware_02/comparison.csv').sort_values('best_validation_selection_loss'))

## Download complete analysis
Checkpoints are included by default because they are required for activation analysis. Set the environment variable to `0` only if the browser cannot download the larger archive.

In [ ]:
from pathlib import Path
from shutil import copytree, make_archive, rmtree
import base64, os
from IPython.display import Javascript, FileLink, display
include_checkpoints = os.environ.get('HAY_EVENT_DOWNLOAD_CHECKPOINTS', '1') == '1'
source = Path('/kaggle/working/hay_micro_event_aware_02')
staging = Path('/kaggle/working/hay_micro_event_aware_02_download')
if staging.exists(): rmtree(staging)
copytree(source, staging, ignore=None if include_checkpoints else lambda path, names: {'checkpoints'} if 'checkpoints' in names else set())
zip_path = Path(make_archive('/kaggle/working/hay_micro_event_aware_02_complete', 'zip', root_dir=staging.parent, base_dir=staging.name))
print('Archive:', zip_path, f'({zip_path.stat().st_size/2**20:.1f} MiB)')
display(FileLink(str(zip_path)))
if zip_path.stat().st_size < 150 * 2**20:
    encoded = base64.b64encode(zip_path.read_bytes()).decode('ascii')
    display(Javascript(f"""const b=atob('{encoded}');const a=new Uint8Array(b.length);for(let i=0;i<b.length;i++)a[i]=b.charCodeAt(i);const u=URL.createObjectURL(new Blob([a],{{type:'application/zip'}}));const x=document.createElement('a');x.href=u;x.download='{zip_path.name}';document.body.appendChild(x);x.click();x.remove();setTimeout(()=>URL.revokeObjectURL(u),60000);"""))
else:
    print('Archivio troppo grande per il download base64: usa il link mostrato sopra o escludi i checkpoint.')